# 03 – Model Training: Median (Random Forest)

**Changes:**
- Replaces neural network (Huber loss) with **Random Forest Regressor**
- More robust to outliers and non-linear relationships
- Model saved for inference in backend

In [33]:
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import os

os.makedirs('../backend/models', exist_ok=True)
np.random.seed(42)

In [34]:
# Load preprocessed data (already scaled and split)
X_train = np.load('../data/X_train.npy')
y_train = np.load('../data/y_train.npy')
X_val = np.load('../data/X_val.npy')
y_val = np.load('../data/y_val.npy')

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

X_train shape: (13667, 13), y_train shape: (13667,)
X_val shape: (2929, 13), y_val shape: (2929,)


In [35]:
# Train Random Forest Regressor
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print("Random Forest training completed.")

Random Forest training completed.


In [36]:
# Save model for inference
joblib.dump(model, '../backend/models/yield_model_median_rf.pkl')
print("Model saved to ../backend/models/yield_model_median_rf.pkl")

Model saved to ../backend/models/yield_model_median_rf.pkl


In [37]:
# Evaluate on validation set
y_pred = model.predict(X_val)
r2 = r2_score(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("=" * 50)
print("Random Forest Median Model Performance (Validation)")
print("=" * 50)
print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.2f} tons/ha")
print(f"RMSE: {rmse:.2f} tons/ha")

Random Forest Median Model Performance (Validation)
R²: 0.8130
MAE: 1.05 tons/ha
RMSE: 4.58 tons/ha


In [38]:
# Optional: feature importance (if needed later)
feature_names = [
    'Crop_target_enc', 'Season_Kharif', 'Season_Rabi', 'Season_Summer',
    'Season_Whole Year', 'Season_Winter', 'State', 'Annual_Rainfall',
    'Fertilizer_per_ha', 'Pesticide_per_ha', 'log_Area', 'Nitrogen', 'Soil Ph'
]
importances = model.feature_importances_
for name, imp in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:5]:
    print(f"{name}: {imp:.3f}")

Crop_target_enc: 0.493
log_Area: 0.218
Fertilizer_per_ha: 0.100
State: 0.072
Annual_Rainfall: 0.044


In [39]:
# Quick inference test
sample = X_val[0:1]
pred = model.predict(sample)[0]
print(f"Sample actual yield: {y_val[0]:.2f}")
print(f"Predicted yield: {pred:.2f}")

Sample actual yield: 0.29
Predicted yield: 0.32
